<a href="https://colab.research.google.com/github/Felix-Leonel/T2-Trabalho-Final-PLN/blob/main/Trabalho_Final_PLN_Detec%C3%A7%C3%A3o_de_Alucina%C3%A7%C3%B5es.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avaliação de Alucinação em LLMs com SQuAD-PT

O contexto do SQuAD-PT é tratado como contexto recuperado em um sistema pseudo-RAG. Para cada exemplo, o modelo recebe um contexto em português, uma pergunta e uma instrução para responder apenas com base no contexto.

As respostas são avaliadas por métricas automáticas de similaridade semântica, TF-IDF e taxa de respostas suspeitas de alucinação.


## Instalação das bibliotecas

Nesta etapa são instaladas as bibliotecas usadas no experimento.

- `datasets`: carregamento do SQuAD-PT;
- `transformers`: carregamento dos modelos geradores;
- `sentence-transformers`: cálculo de similaridade semântica;
- `scikit-learn`: cálculo de TF-IDF;
- `torch`: execução dos modelos;
- `pandas` e `numpy`: manipulação dos dados.

In [ ]:
!pip install -q -U datasets transformers accelerate bitsandbytes sentencepiece
!pip install -q sentence-transformers pandas numpy tqdm scikit-learn torch spacy ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00


## Importação das bibliotecas

As bibliotecas abaixo permitem carregar o dataset, executar os modelos, gerar respostas e calcular as métricas de avaliação.

In [ ]:
import gc
import re
import warnings
import ftfy
import pandas as pd
import numpy as np
import torch
import spacy

from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoProcessor,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

## Configuração do experimento

Nesta etapa são definidos os principais parâmetros do experimento.

### Modelos avaliados

O notebook foi preparado para trabalhar com cinco modelos diferentes:

- `Qwen/Qwen2.5-7B-Instruct`;
- `HuggingFaceTB/SmolLM2-1.7B-Instruct`;
- `TucanoBR/Tucano-2b4`;
- `CEIA-UFG/Gemma-3-Gaia-PT-BR-4b-it`;
- `maritaca-ai/sabia-7b`.

Os modelos Qwen, SmolLM2 e Gaia são tratados como modelos instrucionais. O Tucano e o Sabiá recebem um prompt com exemplos few-shot, pois são modelos base.

### Parâmetros de geração

- `BATCH_SIZE`: quantidade de perguntas processadas por vez;
- `MAX_INPUT_TOKENS`: limite total da entrada;
- `MAX_CONTEXT_TOKENS`: limite do contexto recuperado;
- `MAX_NEW_TOKENS`: limite da resposta gerada;
- `USAR_QUANTIZACAO_4BIT`: reduz o consumo de memória da GPU.

### Parâmetros da recuperação

O contexto é dividido em chunks menores e os trechos mais relevantes são selecionados por uma combinação de similaridade semântica e TF-IDF.

In [ ]:
MODELOS_GERADORES = {
    "Qwen2.5-7B-Instruct": "Qwen/Qwen2.5-7B-Instruct",
    "SmolLM2-1.7B-Instruct": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "Tucano-2b4": "TucanoBR/Tucano-2b4",
    "Gaia-4B": "CEIA-UFG/Gemma-3-Gaia-PT-BR-4b-it",
    "Sabia-7B": "maritaca-ai/sabia-7b",
}

MODELOS_ATIVOS = list(MODELOS_GERADORES.keys())

MODELOS_SELECIONADOS = {
    nome: MODELOS_GERADORES[nome]
    for nome in MODELOS_ATIVOS
}

EMBEDDING_MODEL = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

BATCH_SIZE = 4
MAX_INPUT_TOKENS = 1536
MAX_CONTEXT_TOKENS = 950
MAX_NEW_TOKENS = 72

CHUNK_WORDS = 90
CHUNK_OVERLAP = 0.30
TOP_K_CHUNKS = 3
PESO_SEMANTICO = 0.70
PESO_LEXICAL = 0.30

USAR_QUANTIZACAO_4BIT = True

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Dispositivo:", device)
print("Modelos selecionados:")
for nome, model_id in MODELOS_SELECIONADOS.items():
    print(f"- {nome}: {model_id}")

Dispositivo: cuda
Modelos selecionados:
- Qwen2.5-7B-Instruct: Qwen/Qwen2.5-7B-Instruct
- SmolLM2-1.7B-Instruct: HuggingFaceTB/SmolLM2-1.7B-Instruct
- Tucano-2b4: TucanoBR/Tucano-2b4
- Gaia-4B: CEIA-UFG/Gemma-3-Gaia-PT-BR-4b-it
- Sabia-7B: maritaca-ai/sabia-7b


## Carregamento do SQuAD-PT

O dataset utilizado é o `tgsc/squad-pt-v1.1`, uma versão em português do SQuAD v1.1.

Cada instância contém:

- `title`: tópico ou artigo de origem;
- `context`: texto usado como contexto;
- `question`: pergunta;
- `answers`: resposta correta.

Neste notebook, a coluna `title` é renomeada para `topico`, pois ela representa melhor o tema ou artigo de origem do SQuAD.

In [ ]:
N_EXEMPLOS = 250
EXEMPLOS_POR_TOPICO = 50

# N_EXEMPLOS = 20
# EXEMPLOS_POR_TOPICO = 4

if N_EXEMPLOS % EXEMPLOS_POR_TOPICO != 0:
    raise ValueError("N_EXEMPLOS precisa ser divisível por EXEMPLOS_POR_TOPICO.")

N_TOPICOS = N_EXEMPLOS // EXEMPLOS_POR_TOPICO

dataset = load_dataset("tgsc/squad-pt-v1.1", split="validation")

df_raw = pd.DataFrame(dataset)

def extrair_resposta(answers):
    if isinstance(answers, dict) and len(answers.get("text", [])) > 0:
        return answers["text"][0]
    return ""

df_raw["resposta_referencia"] = df_raw["answers"].apply(extrair_resposta)

df_preparado = df_raw.rename(columns={
    "title": "topico",
    "context": "contexto_referencia",
    "question": "pergunta"
})

df_preparado = df_preparado[
    [
        "id",
        "topico",
        "contexto_referencia",
        "pergunta",
        "resposta_referencia"
    ]
]

df_preparado = df_preparado.dropna()
df_preparado = df_preparado[df_preparado["resposta_referencia"].str.strip() != ""]

topicos_validos = (
    df_preparado["topico"]
    .value_counts()
    .loc[lambda x: x >= EXEMPLOS_POR_TOPICO]
    .index
    .tolist()
)

if len(topicos_validos) < N_TOPICOS:
    raise ValueError(
        f"Não há tópicos suficientes com pelo menos {EXEMPLOS_POR_TOPICO} exemplos. "
        f"Tópicos disponíveis: {len(topicos_validos)}; tópicos necessários: {N_TOPICOS}."
    )

topicos_escolhidos = (
    pd.Series(topicos_validos)
    .sample(n=N_TOPICOS, random_state=42)
    .tolist()
)

df = (
    df_preparado[df_preparado["topico"].isin(topicos_escolhidos)]
    .groupby("topico", group_keys=False)
    .sample(n=EXEMPLOS_POR_TOPICO, random_state=42)
    .reset_index(drop=True)
)

print("Colunas finais:")
print(df.columns)

print("\nTotal de exemplos:", len(df))

print("\nTópicos selecionados:")
print(df["topico"].value_counts())

df.head()

README.md:   0%|          | 0.00/753 [00:00<?, ?B/s]

data/train-00000-of-00001-a07673500b271b(…):   0%|          | 0.00/20.2M [00:00<?, ?B/s]

data/validation-00000-of-00001-d982b2343(…):   0%|          | 0.00/2.68M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87510 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Colunas finais:
Index(['id', 'topico', 'contexto_referencia', 'pergunta',
       'resposta_referencia'],
      dtype='object')

Total de exemplos: 250

Tópicos selecionados:
topico
Computational_complexity_theory    50
Harvard_University                 50
Imperialism                        50
Packet_switching                   50
Scottish_Parliament                50
Name: count, dtype: int64


,id,topico,contexto_referencia,pergunta,resposta_referencia
0,56e1b754cd28a01900c67abf,Computational_complexity_theory,Definições análogas podem ser feitas para requ...,A complexidade da comunicação é um exemplo de ...,medidas de complexidade
1,56e1c720e3433e140042316e,Computational_complexity_theory,Para as classes de complexidade definidas dess...,Que tipo de afirmação é feita no esforço de es...,declarações quantitativas
2,56e1ddfce3433e14004231d6,Computational_complexity_theory,A questão de se P é igual a NP é uma das quest...,Qual é um problema específico da biologia que ...,previsão da estrutura da proteína
3,56e1f10ee3433e1400423225,Computational_complexity_theory,"Da mesma forma, não se sabe se L (o conjunto d...",Quais são as duas classes de complexidade entr...,NL e NC
4,56e1aff7cd28a01900c67a6a,Computational_complexity_theory,Uma máquina de Turing determinística é a máqui...,Qual é o termo usado para identificar uma máqu...,Uma máquina de Turing probabilística


## Carregamento do modelo de similaridade semântica

Para medir similaridade semântica, é utilizado o modelo multilíngue `paraphrase-multilingual-MiniLM-L12-v2`.

Essa métrica compara o significado da resposta gerada pelo modelo com a resposta de referência do dataset.

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device=device)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Pré-processamento textual e pseudo-recuperação

O texto do contexto passa por limpeza, segmentação em chunks com sobreposição e recuperação dos trechos mais relevantes para cada pergunta.

Como o SQuAD-PT já fornece o contexto correto, a recuperação é feita localmente por similaridade de cosseno entre embeddings, substituindo uma base vetorial externa.


In [ ]:
def carregar_spacy_pt():
    try:
        return spacy.load("pt_core_news_sm")
    except Exception:
        nlp = spacy.blank("pt")
        if "sentencizer" not in nlp.pipe_names:
            nlp.add_pipe("sentencizer")
        return nlp


nlp = carregar_spacy_pt()


def preprocessar_texto(texto):
    texto = str(texto)
    texto = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", texto)
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)
    texto = re.sub(r"\S+@\S+", " ", texto)
    texto = re.sub(r"(?m)^\s*\d+\s*$", " ", texto)
    texto = texto.replace("\r", "\n")
    texto = re.sub(r"\n+", " ", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


def chunk_por_sentencas(texto, tamanho_chunk_palavras=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    texto = preprocessar_texto(texto)
    doc = nlp(texto)
    sentencas = [s.text.strip() for s in doc.sents if s.text.strip()]

    chunks, atual, n_atual = [], [], 0
    for sentenca in sentencas:
        n = len(sentenca.split())
        if atual and n_atual + n > tamanho_chunk_palavras:
            chunks.append(" ".join(atual))

            alvo = max(1, int(tamanho_chunk_palavras * overlap))
            mantidas, cont = [], 0
            for s in reversed(atual):
                mantidas.insert(0, s)
                cont += len(s.split())
                if cont >= alvo:
                    break
            atual = mantidas
            n_atual = sum(len(s.split()) for s in atual)

        atual.append(sentenca)
        n_atual += n

    if atual:
        chunks.append(" ".join(atual))
    return chunks or [texto]


def tokens_lexicais(texto):
    return set(re.findall(r"(?u)\b\w{2,}\b", str(texto).lower()))


def score_lexical(pergunta, chunk):
    q = tokens_lexicais(pergunta)
    c = tokens_lexicais(chunk)
    return len(q & c) / max(1, len(q))


def recuperar_chunks_relevantes(pergunta, chunks, embedder, top_k=TOP_K_CHUNKS):
    """Recuperação híbrida: similaridade semântica + correspondência lexical."""
    if not chunks:
        return ""

    embeddings = embedder.encode(
        [str(pergunta)] + list(chunks),
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=32
    )
    scores_sem = cosine_similarity(embeddings[0:1], embeddings[1:])[0]
    scores_lex = np.array([score_lexical(pergunta, c) for c in chunks])

    def minmax(x):
        x = np.asarray(x, dtype=float)
        amplitude = x.max() - x.min()
        return np.zeros_like(x) if amplitude == 0 else (x - x.min()) / amplitude

    scores = PESO_SEMANTICO * minmax(scores_sem) + PESO_LEXICAL * minmax(scores_lex)
    indices = np.argsort(scores)[::-1][:min(top_k, len(chunks))]

    return "\n\n".join(
        f"[Trecho {pos + 1}] {chunks[i]}"
        for pos, i in enumerate(indices)
    )


df["contexto_limpo"] = df["contexto_referencia"].apply(preprocessar_texto)
df["chunks_contexto"] = df["contexto_limpo"].apply(chunk_por_sentencas)

df["contexto_recuperado"] = df.apply(
    lambda row: recuperar_chunks_relevantes(
        row["pergunta"], row["chunks_contexto"], embedder
    ),
    axis=1
)

df[["id", "topico", "pergunta", "resposta_referencia", "contexto_recuperado"]].head()


,id,topico,pergunta,resposta_referencia,contexto_recuperado
0,56e1b754cd28a01900c67abf,Computational_complexity_theory,A complexidade da comunicação é um exemplo de ...,medidas de complexidade,[Trecho 1] Definições análogas podem ser feita...
1,56e1c720e3433e140042316e,Computational_complexity_theory,Que tipo de afirmação é feita no esforço de es...,declarações quantitativas,[Trecho 1] Para as classes de complexidade def...
2,56e1ddfce3433e14004231d6,Computational_complexity_theory,Qual é um problema específico da biologia que ...,previsão da estrutura da proteína,[Trecho 1] A questão de se P é igual a NP é um...
3,56e1f10ee3433e1400423225,Computational_complexity_theory,Quais são as duas classes de complexidade entr...,NL e NC,"[Trecho 1] Da mesma forma, não se sabe se L (o..."
4,56e1aff7cd28a01900c67a6a,Computational_complexity_theory,Qual é o termo usado para identificar uma máqu...,Uma máquina de Turing probabilística,[Trecho 1] Uma máquina de Turing determinístic...


## Construção dos prompts para diferentes tipos de modelo

Os modelos avaliados não usam exatamente o mesmo formato de entrada.

### Modelos instrucionais

Qwen, SmolLM2 e Gaia recebem um prompt de conversa.

### Modelos base

Tucano-2b4 e Sabiá-7B recebem um prompt few-shot com três exemplos. Esses exemplos mostram ao modelo que ele deve completar apenas uma resposta curta.

A geração é feita em lotes para reduzir o tempo de execução. O contexto é truncado por tokens, e não por caracteres, evitando cortes inadequados.

In [ ]:
def identificar_tipo_modelo(model_id):
    nome = model_id.lower()

    if "gemma-3" in nome or "gaia" in nome:
        return "gemma3"

    if "qwen" in nome or "smollm" in nome:
        return "chat"

    if "tucano" in nome or "sabia" in nome:
        return "base"

    return "chat"


def truncar_contexto_por_tokens(
    tokenizer,
    contexto,
    max_tokens=MAX_CONTEXT_TOKENS
):
    ids = tokenizer.encode(
        str(contexto),
        add_special_tokens=False
    )

    if len(ids) <= max_tokens:
        return str(contexto)

    ids = ids[:max_tokens]

    return tokenizer.decode(
        ids,
        skip_special_tokens=True
    )


def criar_conteudo_instrucional(contexto, pergunta):
    return (
        "Responda à pergunta usando exclusivamente as informações "
        "presentes no contexto.\n\n"
        "Regras:\n"
        "1. Forneça somente uma resposta curta e direta.\n"
        "2. Não explique o raciocínio.\n"
        "3. Não repita a pergunta.\n"
        "4. Não acrescente informações externas.\n"
        "5. Use, sempre que possível, as palavras do contexto.\n\n"
        f"CONTEXTO:\n{contexto}\n\n"
        f"PERGUNTA:\n{pergunta}"
    )


def criar_mensagens_chat(contexto, pergunta):
    return [
        {
            "role": "user",
            "content": criar_conteudo_instrucional(
                contexto,
                pergunta
            )
        }
    ]


def criar_mensagens_gemma(contexto, pergunta):
    return [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": criar_conteudo_instrucional(
                        contexto,
                        pergunta
                    )
                }
            ]
        }
    ]


def criar_prompt_base(contexto, pergunta):
    return f"""Complete apenas a resposta final com uma expressão curta retirada do contexto.

Contexto: O Brasil tem Brasília como sua capital.
Pergunta: Qual é a capital do Brasil?
Resposta: Brasília

Contexto: A água congela à temperatura de zero graus Celsius.
Pergunta: Em qual temperatura a água congela?
Resposta: zero graus Celsius

Contexto: Alan Turing propôs a máquina de Turing como um modelo abstrato de computação.
Pergunta: Quem propôs a máquina de Turing?
Resposta: Alan Turing

Contexto: {contexto}
Pergunta: {pergunta}
Resposta:"""


def preparar_prompts_lote(
    model_id,
    tokenizer,
    contextos,
    perguntas,
    processor=None
):
    tipo = identificar_tipo_modelo(model_id)
    prompts = []

    for contexto, pergunta in zip(contextos, perguntas):
        contexto = truncar_contexto_por_tokens(
            tokenizer,
            contexto
        )

        if tipo == "gemma3":
            mensagens = criar_mensagens_gemma(
                contexto,
                pergunta
            )

            prompt = processor.apply_chat_template(
                mensagens,
                tokenize=False,
                add_generation_prompt=True
            )

        elif tipo == "chat":
            mensagens = criar_mensagens_chat(
                contexto,
                pergunta
            )

            prompt = tokenizer.apply_chat_template(
                mensagens,
                tokenize=False,
                add_generation_prompt=True
            )

        else:
            prompt = criar_prompt_base(
                contexto,
                pergunta
            )

        prompts.append(prompt)

    return prompts


def tokenizar_prompts_lote(
    prompts,
    tokenizer,
    tipo_modelo,
    processor=None
):
    if tipo_modelo == "gemma3":
        return processor(
            text=prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
            add_special_tokens=False
        )

    return tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
        add_special_tokens=False
    )


def limpar_resposta_gerada(texto):
    if texto is None:
        return ""

    texto = ftfy.fix_text(str(texto))

    texto = (
        texto
        .replace("Ġ", " ")
        .replace("Ċ", "\n")
        .replace("▁", " ")
    )

    texto = re.sub(
        r"<think>.*?</think>",
        "",
        texto,
        flags=re.IGNORECASE | re.DOTALL
    )

    if "</think>" in texto:
        texto = texto.split("</think>", 1)[-1]

    texto = texto.strip()

    texto = re.sub(
        r"^(resposta final|resposta|answer)\s*:\s*",
        "",
        texto,
        flags=re.IGNORECASE
    )

    texto = re.split(
        r"\n\s*(contexto|pergunta|resposta)\s*:",
        texto,
        maxsplit=1,
        flags=re.IGNORECASE
    )[0]

    linhas = [
        linha.strip()
        for linha in texto.splitlines()
        if linha.strip()
    ]

    if linhas:
        texto = linhas[0]

    texto = re.sub(r"\s+", " ", texto)

    return texto.strip(' "\'“”‘’')


def criar_configuracao_quantizacao():
    if not (
        USAR_QUANTIZACAO_4BIT
        and torch.cuda.is_available()
    ):
        return None

    dtype_compute = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype_compute
    )


def carregar_modelo(model_id):
    tipo = identificar_tipo_modelo(model_id)
    quantization_config = criar_configuracao_quantizacao()

    kwargs_modelo = {
        "device_map": "auto",
        "trust_remote_code": True,
        "low_cpu_mem_usage": True
    }

    if quantization_config is not None:
        kwargs_modelo["quantization_config"] = quantization_config
    elif torch.cuda.is_available():
        kwargs_modelo["torch_dtype"] = (
            torch.bfloat16
            if torch.cuda.is_bf16_supported()
            else torch.float16
        )

    processor = None

    if tipo == "gemma3":
        processor = AutoProcessor.from_pretrained(
            model_id,
            trust_remote_code=True
        )
        tokenizer = processor.tokenizer
    else:
        tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            trust_remote_code=True,
            use_fast=True
        )

    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            tokenizer.add_special_tokens(
                {"pad_token": "<|pad|>"}
            )

    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        **kwargs_modelo
    )

    if (
        model.get_input_embeddings().num_embeddings
        < len(tokenizer)
    ):
        model.resize_token_embeddings(len(tokenizer))

    model.eval()

    return model, tokenizer, processor


def mover_inputs_para_dispositivo(inputs, device_modelo):
    return {
        chave: valor.to(device_modelo)
        if isinstance(valor, torch.Tensor)
        else valor
        for chave, valor in inputs.items()
    }


def gerar_respostas_lote(
    model,
    tokenizer,
    model_id,
    contextos,
    perguntas,
    processor=None
):
    tipo = identificar_tipo_modelo(model_id)

    prompts = preparar_prompts_lote(
        model_id=model_id,
        tokenizer=tokenizer,
        contextos=contextos,
        perguntas=perguntas,
        processor=processor
    )

    inputs = tokenizar_prompts_lote(
        prompts=prompts,
        tokenizer=tokenizer,
        tipo_modelo=tipo,
        processor=processor
    )

    device_modelo = next(model.parameters()).device

    inputs = mover_inputs_para_dispositivo(
        inputs,
        device_modelo
    )

    comprimento_entrada = inputs["input_ids"].shape[1]

    parametros = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
        "repetition_penalty": 1.05,
        "pad_token_id": tokenizer.pad_token_id,
        "use_cache": True
    }

    if tipo in {"chat", "gemma3"}:
        parametros["no_repeat_ngram_size"] = 3

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            **parametros
        )

    respostas = []

    for output_ids in outputs:
        novos_tokens = output_ids[comprimento_entrada:]

        texto_bruto = tokenizer.decode(
            novos_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        respostas.append(
            limpar_resposta_gerada(texto_bruto)
        )

    return respostas


## Geração das respostas

Nesta etapa, cada modelo é carregado separadamente e usado para responder às perguntas.

A função de geração:

1. identifica o tipo de modelo;
2. aplica o prompt correto;
3. processa as perguntas em lotes;
4. tenta gerar individualmente quando um lote apresenta erro;
5. libera a memória antes de carregar o próximo modelo.

A separação dos tokens gerados é feita usando o comprimento real da entrada tokenizada. Isso evita que o texto do prompt apareça dentro de `resposta_modelo`.

Também não é definido manualmente um único `eos_token_id`, porque alguns modelos possuem mais de um token de finalização.

In [ ]:
def gerar_respostas_modelo(
    model,
    tokenizer,
    model_id,
    dataframe,
    processor=None
):
    respostas = []

    for inicio in tqdm(
        range(0, len(dataframe), BATCH_SIZE),
        desc=f"Gerando com {model_id.split('/')[-1]}"
    ):
        fim = inicio + BATCH_SIZE
        lote = dataframe.iloc[inicio:fim]

        contextos = lote["contexto_recuperado"].tolist()
        perguntas = lote["pergunta"].tolist()

        try:
            respostas_lote = gerar_respostas_lote(
                model=model,
                tokenizer=tokenizer,
                model_id=model_id,
                contextos=contextos,
                perguntas=perguntas,
                processor=processor
            )

        except Exception as erro_lote:
            print(
                f"\nErro no lote {inicio}:{fim}: "
                f"{type(erro_lote).__name__}: {erro_lote}"
            )
            print("Tentando os exemplos individualmente.")

            respostas_lote = []

            for contexto, pergunta in zip(
                contextos,
                perguntas
            ):
                try:
                    resposta = gerar_respostas_lote(
                        model=model,
                        tokenizer=tokenizer,
                        model_id=model_id,
                        contextos=[contexto],
                        perguntas=[pergunta],
                        processor=processor
                    )[0]

                except Exception as erro_individual:
                    print(
                        "Erro individual:",
                        type(erro_individual).__name__,
                        erro_individual
                    )
                    resposta = ""

                respostas_lote.append(resposta)

        respostas.extend(respostas_lote)

    return respostas


todas_respostas = []

for nome_modelo, model_id in MODELOS_SELECIONADOS.items():
    print("\n" + "=" * 80)
    print("Carregando modelo:", nome_modelo)
    print("Identificador:", model_id)
    print("Tipo:", identificar_tipo_modelo(model_id))
    print("=" * 80)

    model = None
    tokenizer = None
    processor = None

    try:
        model, tokenizer, processor = carregar_modelo(
            model_id
        )

        respostas = gerar_respostas_modelo(
            model=model,
            tokenizer=tokenizer,
            model_id=model_id,
            dataframe=df,
            processor=processor
        )

        df_modelo = df.copy()
        df_modelo["modelo"] = nome_modelo
        df_modelo["resposta_modelo"] = respostas

        todas_respostas.append(df_modelo)

        total_vazias = (
            df_modelo["resposta_modelo"]
            .fillna("")
            .str.strip()
            .eq("")
            .sum()
        )

        print(
            f"Respostas vazias: "
            f"{total_vazias}/{len(df_modelo)}"
        )

    except Exception as erro_modelo:
        print(
            f"Falha no modelo {nome_modelo}: "
            f"{type(erro_modelo).__name__}: {erro_modelo}"
        )

    finally:
        if model is not None:
            del model

        if tokenizer is not None:
            del tokenizer

        if processor is not None:
            del processor

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


if not todas_respostas:
    raise RuntimeError(
        "Nenhum modelo foi executado com sucesso."
    )

df_respostas = pd.concat(
    todas_respostas,
    ignore_index=True
)

df_respostas["resposta_vazia"] = (
    df_respostas["resposta_modelo"]
    .fillna("")
    .str.strip()
    .eq("")
)

display(
    df_respostas[
        [
            "id",
            "topico",
            "modelo",
            "contexto_referencia",
            "contexto_limpo",
            "contexto_recuperado",
            "pergunta",
            "resposta_referencia",
            "resposta_modelo"
        ]
    ].head()
)

display(
    df_respostas
    .groupby("modelo")["resposta_vazia"]
    .agg(["sum", "count", "mean"])
)


Carregando modelo: Qwen2.5-7B-Instruct
Identificador: Qwen/Qwen2.5-7B-Instruct
Tipo: chat


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Gerando com sabia-7b:   0%|          | 0/63 [00:00<?, ?it/s]

Respostas vazias: 0/250


,id,topico,modelo,contexto_referencia,contexto_limpo,contexto_recuperado,pergunta,resposta_referencia,resposta_modelo
0,56e1b754cd28a01900c67abf,Computational_complexity_theory,Qwen2.5-7B-Instruct,Definições análogas podem ser feitas para requ...,Definições análogas podem ser feitas para requ...,[Trecho 1] Definições análogas podem ser feita...,A complexidade da comunicação é um exemplo de ...,medidas de complexidade,medida de complexity
1,56e1c720e3433e140042316e,Computational_complexity_theory,Qwen2.5-7B-Instruct,Para as classes de complexidade definidas dess...,Para as classes de complexidade definidas dess...,[Trecho 1] Para as classes de complexidade def...,Que tipo de afirmação é feita no esforço de es...,declarações quantitativas,declaraçõesquantitativas
2,56e1ddfce3433e14004231d6,Computational_complexity_theory,Qwen2.5-7B-Instruct,A questão de se P é igual a NP é uma das quest...,A questão de se P é igual a NP é uma das quest...,[Trecho 1] A questão de se P é igual a NP é um...,Qual é um problema específico da biologia que ...,previsão da estrutura da proteína,previsão daestrutura da protéina
3,56e1f10ee3433e1400423225,Computational_complexity_theory,Qwen2.5-7B-Instruct,"Da mesma forma, não se sabe se L (o conjunto d...","Da mesma forma, não se sabe se L (o conjunto d...","[Trecho 1] Da mesma forma, não se sabe se L (o...",Quais são as duas classes de complexidade entr...,NL e NC,NL e NC
4,56e1aff7cd28a01900c67a6a,Computational_complexity_theory,Qwen2.5-7B-Instruct,Uma máquina de Turing determinística é a máqui...,Uma máquina de Turing determinística é a máqui...,[Trecho 1] Uma máquina de Turing determinístic...,Qual é o termo usado para identificar uma máqu...,Uma máquina de Turing probabilística,máquina de Turing probabilidade


,sum,count,mean
modelo,,,
Gaia-4B,0,250,0.0
Qwen2.5-7B-Instruct,0,250,0.0
Sabia-7B,0,250,0.0
SmolLM2-1.7B-Instruct,0,250,0.0
Tucano-2b4,0,250,0.0


## Similaridade semântica

A similaridade semântica mede se a resposta gerada possui significado próximo da resposta de referência, mesmo que as palavras utilizadas sejam diferentes.

Para isso:

1. a resposta do modelo e a resposta correta são transformadas em embeddings;
2. é calculada a similaridade do cosseno entre os vetores;
3. valores mais altos indicam maior proximidade semântica.

In [ ]:
def semantic_similarity(a, b):
    a = str(a).strip()
    b = str(b).strip()

    if not a or not b:
        return 0.0

    emb = embedder.encode(
        [a, b],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return float(
        np.dot(emb[0], emb[1])
    )


df_respostas["semantic_similarity"] = df_respostas.apply(
    lambda row: semantic_similarity(
        row["resposta_modelo"],
        row["resposta_referencia"]
    ),
    axis=1
)

display(
    df_respostas[
        [
            "modelo",
            "pergunta",
            "resposta_referencia",
            "resposta_modelo",
            "semantic_similarity"
        ]
    ].head()
)

,modelo,pergunta,resposta_referencia,resposta_modelo,semantic_similarity
0,Qwen2.5-7B-Instruct,A complexidade da comunicação é um exemplo de ...,medidas de complexidade,medida de complexity,0.966876
1,Qwen2.5-7B-Instruct,Que tipo de afirmação é feita no esforço de es...,declarações quantitativas,declaraçõesquantitativas,0.827053
2,Qwen2.5-7B-Instruct,Qual é um problema específico da biologia que ...,previsão da estrutura da proteína,previsão daestrutura da protéina,0.924592
3,Qwen2.5-7B-Instruct,Quais são as duas classes de complexidade entr...,NL e NC,NL e NC,1.000000
4,Qwen2.5-7B-Instruct,Qual é o termo usado para identificar uma máqu...,Uma máquina de Turing probabilística,máquina de Turing probabilidade,0.920615


## Similaridade TF-IDF

Nesta etapa é calculada uma métrica de TF-IDF.

A métrica compara a resposta gerada pelo modelo com o contexto recuperado usado como evidência. Valores mais altos indicam maior sobreposição lexical entre a resposta e o contexto; valores baixos podem indicar menor aderência ao conteúdo fornecido e maior risco de alucinação.

In [ ]:
def tfidf_similarity(a, b):
    try:
        vectorizer = TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            sublinear_tf=True
        )
        X = vectorizer.fit_transform([str(a), str(b)])
        return float(cosine_similarity(X[0:1], X[1:2])[0][0])
    except ValueError:
        return 0.0


def sentencas_do_contexto(texto):
    texto = re.sub(r"\[Trecho \d+\]", " ", str(texto))
    doc = nlp(texto)
    return [s.text.strip() for s in doc.sents if s.text.strip()]


def tfidf_suporte_local(resposta, contexto):
    resposta = str(resposta).strip()
    if not resposta:
        return 0.0
    sentencas = sentencas_do_contexto(contexto)
    if not sentencas:
        return 0.0
    return max(tfidf_similarity(resposta, s) for s in sentencas)

df_respostas["tfidf_contexto_global"] = df_respostas.apply(
    lambda row: tfidf_similarity(row["resposta_modelo"], row["contexto_recuperado"]),
    axis=1
)

df_respostas["tfidf_contexto"] = df_respostas.apply(
    lambda row: tfidf_suporte_local(row["resposta_modelo"], row["contexto_recuperado"]),
    axis=1
)

df_respostas[["modelo", "resposta_modelo", "tfidf_contexto_global", "tfidf_contexto"]].head()


,modelo,resposta_modelo,tfidf_contexto_global,tfidf_contexto
0,Qwen2.5-7B-Instruct,medida de complexity,0.118606,0.155285
1,Qwen2.5-7B-Instruct,declaraçõesquantitativas,0.000000,0.000000
2,Qwen2.5-7B-Instruct,previsão daestrutura da protéina,0.103133,0.140974
3,Qwen2.5-7B-Instruct,NL e NC,0.122617,0.123849
4,Qwen2.5-7B-Instruct,máquina de Turing probabilidade,0.201260,0.374316


## Normalização das métricas

As métricas são normalizadas com Min-Max para ficarem no intervalo entre 0 e 1.

O `final_score_minmax` é calculado pela soma da similaridade semântica normalizada e do TF-IDF normalizado. Valores maiores indicam melhor desempenho agregado.

In [ ]:
metricas = [
    "semantic_similarity",
    "tfidf_contexto"
]

for metrica in metricas:

    min_val = df_respostas[metrica].min()
    max_val = df_respostas[metrica].max()

    if max_val - min_val == 0:
        df_respostas[f"{metrica}_minmax"] = 0
    else:
        df_respostas[f"{metrica}_minmax"] = (
            df_respostas[metrica] - min_val
        ) / (max_val - min_val)


df_respostas["final_score_minmax"] = (
    df_respostas["semantic_similarity_minmax"] +
    df_respostas["tfidf_contexto_minmax"]
)

df_respostas[["modelo", "final_score_minmax"]].head()

,modelo,final_score_minmax
0,Qwen2.5-7B-Instruct,1.126055
1,Qwen2.5-7B-Instruct,0.847387
2,Qwen2.5-7B-Instruct,1.074432
3,Qwen2.5-7B-Instruct,1.123849
4,Qwen2.5-7B-Instruct,1.304265


## Ranking geral dos modelos

Aqui são calculadas as médias das métricas por modelo.

A coluna `label_alucinacao` representa a proporção de respostas classificadas como suspeitas.


In [ ]:
ranking_modelos = df_respostas.groupby("modelo")[
    [
        "semantic_similarity",
        "tfidf_contexto",
        "tfidf_contexto_global",
        "final_score_minmax"
    ]
].mean().sort_values(
    "final_score_minmax",
    ascending=False
)

display(ranking_modelos)

,semantic_similarity,tfidf_contexto,tfidf_contexto_global,label_alucinacao,resposta_vazia,final_score_minmax
modelo,,,,,,
Sabia-7B,0.717600,0.318774,0.162298,0.504,0.0,1.069577
Tucano-2b4,0.600180,0.302743,0.174101,0.632,0.0,0.949931
Qwen2.5-7B-Instruct,0.677614,0.130262,0.073187,0.784,0.0,0.845780
Gaia-4B,0.620533,0.164515,0.101345,0.692,0.0,0.829664
SmolLM2-1.7B-Instruct,0.459222,0.099282,0.068220,0.920,0.0,0.622086


## Resultado final resumido

Esta tabela reúne as principais métricas em formato adequado para a seção de resultados do trabalho.


In [ ]:
resultado_final = df_respostas.groupby("modelo").agg(
    similaridade_semantica_media=(
        "semantic_similarity",
        "mean"
    ),
    tfidf_medio=(
        "tfidf_contexto",
        "mean"
    ),
    tfidf_global_medio=(
        "tfidf_contexto_global",
        "mean"
    ),
    score_final_medio=(
        "final_score_minmax",
        "mean"
    )
).reset_index()

display(resultado_final)

,modelo,similaridade_semantica_media,tfidf_medio,tfidf_global_medio,score_final_medio,taxa_alucinacao_%,taxa_resposta_vazia_%
2,Sabia-7B,0.717600,0.318774,0.162298,1.069577,50.4,0.0
4,Tucano-2b4,0.600180,0.302743,0.174101,0.949931,63.2,0.0
1,Qwen2.5-7B-Instruct,0.677614,0.130262,0.073187,0.845780,78.4,0.0
0,Gaia-4B,0.620533,0.164515,0.101345,0.829664,69.2,0.0
3,SmolLM2-1.7B-Instruct,0.459222,0.099282,0.068220,0.622086,92.0,0.0


## Análise por tópico

Esta análise mostra o desempenho médio dos modelos em cada tópico do SQuAD-PT.


In [ ]:
ranking_topico = df_respostas.groupby(
    ["topico", "modelo"]
)[
    [
        "semantic_similarity",
        "tfidf_contexto",
        "final_score_minmax"
    ]
].mean().reset_index()

display(ranking_topico.head(20))

,topico,modelo,semantic_similarity,tfidf_contexto,label_alucinacao,resposta_vazia,final_score_minmax
0,Computational_complexity_theory,Gaia-4B,0.688078,0.194654,0.50,0.0,0.919405
1,Computational_complexity_theory,Qwen2.5-7B-Instruct,0.796769,0.167698,0.56,0.0,0.988361
2,Computational_complexity_theory,Sabia-7B,0.776682,0.285695,0.44,0.0,1.088633
3,Computational_complexity_theory,SmolLM2-1.7B-Instruct,0.528927,0.110655,0.86,0.0,0.694968
4,Computational_complexity_theory,Tucano-2b4,0.610604,0.302927,0.50,0.0,0.959314
5,Harvard_University,Gaia-4B,0.667509,0.125526,0.86,0.0,0.832127
6,Harvard_University,Qwen2.5-7B-Instruct,0.699315,0.072356,0.94,0.0,0.807023
7,Harvard_University,Sabia-7B,0.774206,0.215765,0.54,0.0,1.016518
8,Harvard_University,SmolLM2-1.7B-Instruct,0.478604,0.089174,0.94,0.0,0.629081
9,Harvard_University,Tucano-2b4,0.645343,0.192383,0.72,0.0,0.879425


## Salvamento dos resultados


In [ ]:
df_respostas.to_csv(
    "/content/resultados_squadpt_pseudo_rag_preprocessado_multimodelo.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

ranking_modelos.to_csv(
    "/content/ranking_modelos_squadpt.csv",
    sep=";",
    encoding="utf-8-sig"
)

ranking_topico.to_csv(
    "/content/ranking_topico_squadpt.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

resultado_final.to_csv(
    "/content/resultado_final_squadpt.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

suspeitas.to_csv(
    "/content/respostas_suspeitas_squadpt.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

diagnostico_vazias.to_csv(
    "/content/diagnostico_respostas_vazias.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos salvos em /content.")

Arquivos salvos em /content.
